# **Notebook**: LISA Cluster Analysis

__Content__: Code utilized to run Local Indicator of Spatial Association (LISA) cluster analysis for LED-high-exposure-population and CRE

#### Initial setup

In [ ]:
%pip install libpysal

In [ ]:
%pip install esda

In [ ]:
# Load libraries
import geopandas as gpd
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
from libpysal.weights import KNN
from esda.moran import Moran_Local_BV
from shapely.geometry import Point
import libpysal
from esda.moran import Moran_BV

# For statistical metrics and log transformations
import numpy as np
from scipy.stats import zscore
from scipy.stats import skew, pearsonr, spearmanr
from sklearn.neighbors import NearestNeighbors

# For plotting
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import contextily as ctx # For background basemaps
import matplotlib.pyplot as plt


import os

data_path = "..."

#### Load data

In [ ]:
''' Load datasets'''
# Load Landslide Exposure Database
led_inventory = gpd.read_file(os.path.join(data_path, "process_building_inventory", "US_LandslideExposureDatabase_points.gpkg"))

# load census tracts master dataset
census_tracts_master = gpd.read_file(os.path.join(data_path, "process_building_inventory", "US_census_tracts_master.gpkg"))

## 1) LISA Clusters

In [ ]:
# 1. LOAD DATA
# tracts: GeoDataFrame of census polygons with demographic 'C'
# points: GeoDataFrame of points 'P'
tracts = census_tracts_master[census_tracts_master['high_exp_pop']>0 & census_tracts_master['ALAND']>0].copy()
points = led_inventory[led_inventory['susc_class'] == 'high'].copy()

#### Data preparation:
* Intra-tract DBSCAN clusters & refined census tract centroids
* Log transformation for skew data

In [ ]:
# Initialize a column in your points GDF
points['is_cluster'] = False

def get_refined_centroid(tract_poly, points_gdf):
    # Select points within this tract
    # We use .index to map the results back to the original points_gdf
    tract_points_idx = points_gdf[points_gdf.intersects(tract_poly)].index
    
    if len(tract_points_idx) < 5: 
        return tract_poly.centroid
    
    coords = np.array(list(zip(points_gdf.loc[tract_points_idx].geometry.x, 
                               points_gdf.loc[tract_points_idx].geometry.y)))
    
    # Run DBSCAN
    db = DBSCAN(eps=1000, min_samples=5).fit(coords)
    labels = db.labels_
    
    # Identify which of these local points are part of a cluster (label != -1)
    in_cluster_local_indices = tract_points_idx[labels != -1]
    
    # Update the global points dataframe
    points_gdf.loc[in_cluster_local_indices, 'is_cluster'] = True
    
    if len(labels[labels != -1]) > 0:
        largest_cluster_label = pd.Series(labels[labels != -1]).value_counts().idxmax()
        cluster_coords = coords[labels == largest_cluster_label]
        return Point(cluster_coords.mean(axis=0))
    else:
        return tract_poly.centroid

# Apply the refined centroid logic
tracts['refined_centroid'] = tracts.geometry.apply(lambda poly: get_refined_centroid(poly, points))
print("DBSCAN Clustering Complete. Points in clusters:", points['is_cluster'].sum())

# Get count of buildings per tract for LISA
# tracts['P_count'] = tracts.apply(lambda row: len(points[points.intersects(row.geometry)]), axis=1)
# tracts['P_count'] = tracts['P_count'].fillna(0)

# --- PREPARE DATA FOR LISA ---
print("Number of buildings in high landslide exposure:", tracts['high_exp_bld'].sum())
# 1. Calculate the raw rate
# Avoid division by zero by replacing 0 population with 1 or NaN
tracts['rate'] = tracts['high_exp_pop']
tracts['rate'] = tracts['rate'].fillna(0)

# 2. Log-Normalization (Stabilizes the variance)
# This is the "magic" step to find significant clusters in skewed data
tracts['log_rate'] = np.log1p(tracts['rate'])

# 3. Z-Score Standardization (Centers the data for LISA)
tracts['P_final'] = zscore(tracts['log_rate'])
tracts['C_final'] = zscore(tracts['PRED3_PE'].fillna(0))

# --- INTERMEDIATE CHECKS ---

# --- 1. SKEWNESS AUDIT ---
raw_rate_skew = skew(tracts['rate'].dropna())
log_rate_skew = skew(tracts['log_rate'].dropna())

print(f"Skewness (Fisher-Pearson):")
print(f"  - Raw Exposure Rate: {raw_rate_skew:.4f}")
print(f"  - Log-Transformed Rate: {log_rate_skew:.4f}")

# A successful transform should bring the value closer to 0.
# If log_rate_skew is between -1 and 1, your data is much more 'normal'.

# --- 2. INDEPENDENT CORRELATION AUDIT ---
# We compare the finalized point intensity (P_final) vs. demographic context (C_final)
# This is the non-spatial counterpart to your LISA.

# Pearson (Linear)
corr_p, p_val_p = pearsonr(tracts['P_final'], tracts['C_final'])

# Spearman (Rank - often better for census data)
corr_s, p_val_s = spearmanr(tracts['P_final'], tracts['C_final'])

print(f"\nIndependent Correlation Analysis:")
print(f"  - Pearson r: {corr_p:.4f} (p = {p_val_p:.4f})")
print(f"  - Spearman rho: {corr_s:.4f} (p = {p_val_s:.4f})")

# --- STEP B: KNN WEIGHTS MATRIX ---
# We use the refined centroids to define "neighborhood"
coords = np.array(list(zip(tracts.refined_centroid.x, tracts.refined_centroid.y)))
w = KNN.from_array(coords, k=8) 
w.transform = 'r' # Row-standardize

# --- STEP C: BIVARIATE LISA ---
# Relationship between Point Count (P) and Demographic Indicator (C)
y1 = tracts['P_final'].values
y2 = tracts['C_final'].values

# Calculate Bivariate Local Moran's I
lisa_bi = Moran_Local_BV(y1, y2, w)

# Add results back to tracts for mapping
tracts['lisa_q'] = lisa_bi.q # 1=HH, 2=LH, 3=LL, 4=HL
tracts['lisa_p'] = lisa_bi.p_sim # p-values for significance

print("LISA Analysis Complete. Significant clusters found:", (tracts['lisa_p'] < 0.05).sum())

In [ ]:
# --- PREP: Standard Geometric Centroids ---
# Use the actual polygon centroids
geom_coords = np.array(list(zip(tracts.geometry.centroid.x, tracts.geometry.centroid.y)))
w_std = libpysal.weights.KNN.from_array(geom_coords, k=8)
w_std.transform = 'r'

# --- PREP: DBSCAN Refined Centroids ---
# (Using the 'refined_centroid' column created in the previous step)
refined_coords = np.array(list(zip(tracts.refined_centroid.x, tracts.refined_centroid.y)))
w_ref = libpysal.weights.KNN.from_array(refined_coords, k=8)
w_ref.transform = 'r'

# --- ANALYSIS: Compare Global Bivariate Moran's I ---
y1 = tracts['P_final'].values.astype(float)
y2 = tracts['C_final'].values.astype(float)

# Global Moran's I for Standard approach
moran_std = Moran_BV(y1, y2, w_std)

# Global Moran's I for Your 3-Step approach
moran_ref = Moran_BV(y1, y2, w_ref)

print(f"Global Moran's I (Standard): {moran_std.I:.4f}")
print(f"Global Moran's I (DBSCAN Refined): {moran_ref.I:.4f}")

# Statistical Significance Check
print(f"Standard P-value: {moran_std.p_sim:.4f}")
print(f"Refined P-value: {moran_ref.p_sim:.4f}")

# --- DECISION LOGIC ---
if moran_ref.I > moran_std.I:
    improvement = ((moran_ref.I - moran_std.I) / moran_std.I) * 100
    print(f"\nConclusion: The 3-Step method improved spatial correlation by {improvement:.2f}%.")
else:
    print("\nConclusion: The standard method is sufficient; DBSCAN refinement did not increase correlation.")

#### Visualization

In [ ]:
y1_std = tracts['P_final'].values
y2_std = tracts['C_final'].values

lisa_bi_std = Moran_Local_BV(y1_std, y2_std, w_std)


# 4. Visualization

# 1. SETUP FIGURE
fig, ax = plt.subplots(2, 2, figsize=(15, 10), sharex=True, sharey=True, dpi=300)

# --- LEFT MAP: Step A (DBSCAN & Refined Centroids) ---
# Plot the census tracts as a base
tracts.plot(ax=ax[0,0], color='white', edgecolor='lightgrey')

# Plot all points P in grey
points.plot(ax=ax[0,0], marker='o', color='grey', markersize=2, alpha=0.3, label='Noise/Unclustered')

# Highlight the points that were part of a DBSCAN cluster
# (Assuming you saved cluster labels during Step A)
if points['is_cluster'].sum() > 0:
    clustered_points = points[points['is_cluster'] == True]
    clustered_points.plot(ax=ax[0,0], marker='o', color='red', markersize=5, label='DBSCAN Clusters')
else:
    print("No clustered points to plot.")
    points.plot(ax=ax[0,0], marker='o', color='red', markersize=5, label='High Exposure Buildings')

# Plot the refined centroids to show where the LISA weights are calculated from
# tracts_gdf_centroids = gpd.GeoDataFrame(tracts, geometry='refined_centroid')
# tracts_gdf_centroids.plot(ax=ax[0,0], marker='x', color='blue', markersize=15, label='Refined Centroids')

ax[0,0].set_title("Step A: Intra-Tract Density (DBSCAN)\n& Refined Centroids", fontsize=15)
ax[0,0].legend()

# Add map of tracts['PRED3_PE']
tracts.plot(column='PRED3_PE', ax=ax[0,1], alpha=0.5, legend=True, cmap='Purples', legend_kwds={'label': "CRE 3+", 'orientation': "vertical"})
ax[0,1].set_title("Census Tracts Colored by PRED3_PE", fontsize=15)
ax[0,1].set_axis_off()

# --- RIGHT MAP: Step C (Bivariate LISA Cluster Map) ---
# Using splot to automatically handle the colors for HH, LL, LH, HL
lisa_cluster(lisa_bi, tracts, p=0.05, ax=ax[1,0])

ax[1,0].set_title("Bivariate LISA Map using DBSCAN centroids\n(High Exp Per Capita vs. CRE 3+)", fontsize=15)

# The lisa_cluster function automatically handles the coloring:
# Red: High-High, Blue: Low-Low, Light Blue: Low-High, Pink: High-Low
lisa_cluster(lisa_bi_std, tracts, p=0.05, ax=ax[1,1])

ax[1,1].set_title("Bivariate LISA Cluster Map\n(Standard Geometric Centroids)", fontsize=16)


# Optional: Print out the count of significant clusters
sig_clusters_ref = (lisa_bi.p_sim < 0.05).sum()
sig_clusters_std = (lisa_bi_std.p_sim < 0.05).sum()
print(f"Number of statistically significant clusters (DBSCAN Refined): {sig_clusters_ref}")
print(f"Number of statistically significant clusters (Standard Centroids): {sig_clusters_std}")

# 2. FINALIZE
for a in ax.ravel():
    a.set_axis_off()
    # Optional: Add a basemap if you have contextily installed
    # ctx.add_basemap(a, crs=tracts.crs.to_string(), source=ctx.providers.CartoDB.Positron)

plt.tight_layout()
plt.show()

In [ ]:
results = []
for k_val in [4, 6, 8, 10, 12]:
    # Use your refined centroids from the 3-step method
    w_test = KNN.from_array(refined_coords, k=k_val)
    w_test.transform = 'r'
    
    mi = Moran_BV(y1, y2, w_test)
    results.append({'k': k_val, 'Moran_I': mi.I, 'p_value': mi.p_sim})

# Display the results as a DataFrame
df_results = pd.DataFrame(results)
print(df_results)

In [ ]:

plt.ion()
# Define a custom color map: 0=Grey (NS), 1=Red (HH), 4=Orange (HL)
# We skip 2 and 3 because they are truncated

# 1. Create a mask for "High P" (Quadrants 1 and 4)
# In esda, lisa_bi.q: 1=HH, 2=LH, 3=LL, 4=HL
high_p_mask = (tracts['lisa_q'] == 1) | (tracts['lisa_q'] == 4)

# 2. Create a mask for significance
significant_mask = tracts['lisa_p'] < 0.01

# 3. Create a new column for "Truncated LISA"
# We initialize it as 0 (not significant or not high-P)
tracts['lisa_truncated'] = 0

# Assign HH (1) and HL (4) only where significant
tracts.loc[high_p_mask & significant_mask, 'lisa_truncated'] = tracts['lisa_q']

# 4. Count your new categories
print(tracts['lisa_truncated'].value_counts())

cmap = ListedColormap(['#eeeeee', '#d7191c', '#fdae61'])

from matplotlib.colors import BoundaryNorm
fig, ax = plt.subplots(figsize=(20, 20), dpi=300)

# Map values 0->grey, 1->red, 4->orange using discrete bins
bounds = [-0.5, 0.5, 2.5, 5.0]
norm = BoundaryNorm(bounds, cmap.N)
tracts_plot = tracts[(tracts['ALAND'] > 0)].copy()  # Optional: filter out non-land tracts
tracts_plot.plot(column='lisa_truncated', cmap=cmap, norm=norm, ax=ax, edgecolor='0.8', linewidth=0.01, alpha=0.7)

# Add a custom legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='#d7191c', lw=4, label='High Exposure - High Social Vulnerability'),
    Line2D([0], [0], color='#fdae61', lw=4, label='High Exposure - Low Social Vulnerability'),
    Line2D([0], [0], color='#eeeeee', lw=4, label='Not Significant / Low Building Density')
]

ctx.add_basemap(ax=ax, crs=tracts.crs.to_string(), source=ctx.providers.OpenStreetMap.Mapnik)
ax.legend(handles=legend_elements, loc='lower right')
ax.set_title("Bivariate LISA (Truncated High-High and High-Low)", fontsize=14)
plt.show()

#### Save LISA results for final Census Tracts Master Database

In [ ]:
# Merge LISA results
census_tracts_master = census_tracts_master.merge(tracts[['GEOIDFQ', 'lisa_q', 'lisa_p', 'lisa_truncated']], on='GEOIDFQ', how='left')
census_tracts_master = census_tracts_master.drop(columns=['index_right'], errors='ignore')

# save final tracts with LISA results
out_path = os.path.join(data_path, "process_building_inventory", "US_CensusTractsMaster_Database.gpkg")
census_tracts_master.to_file(out_path, driver="GPKG")

#### End of Script